# Street Scout — Reference Embedding Build on Colab T4

This notebook rebuilds `reference_embeddings.pt` (the CLIP embedding cache used by the
classifier and FAISS fallback) on a GPU instead of your local CPU — the same job
`ai-service`'s `CarClassifier._build_index()` does at startup, just much faster.

**Why:** with the new 6-variant augmentation set, this build is estimated at 35+ hours
on a 4-thread local CPU. On a Colab T4 it should take well under 2 hours.

**v3 update:** reference images are now YOLO-cropped before embedding, matching what
the live service does at inference time (previously a train/inference mismatch — the
classifier trained on uncropped photos but always predicted on cropped ones, which a
held-out evaluation measured as an ~11-point accuracy cost).

**v4 update:** the reference image set was substantially expanded (broad quality-controlled
re-fetch across all 813 generations). This version loads from one fresh, complete zip
instead of the old loose-folder + delta-zip approach, since the dataset changed too much
for an incremental sync to make sense.

**Before running**, upload to `MyDrive/street_scout/`:
- `reference_images_full.zip` (the complete, current reference image set — replaces
  any older loose `reference_images/` folder or delta zip from previous runs)
- `class_labels.json` (from your local `ai-service/model/`)
- `reference_images.json` (from your local `ai-service/model/`)
- `no_car_images.txt` (optional — from `ai-service/scripts/`)
- `clip_lora/` (optional — your trained LoRA adapter, if you have one; keeps the
  embeddings consistent with what your local ai-service actually serves)

Use **Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# Step 1 — check we have a GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Step 2 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3 — install peft (only needed if you're applying a clip_lora adapter)
!pip install -q peft

In [ ]:
# Step 4 — unzip the full reference image archive from Drive (~12k images, a
# couple of minutes). Unzipping happens here on Linux, which has no issue with
# filenames containing characters like ':' that are problematic on Windows.
import zipfile, os

DRIVE_DIR = '/content/drive/MyDrive/street_scout'   # class_labels.json, reference_images.json, the zip, etc.
IMAGES_DIR = '/content/reference_images'

full_zip = f'{DRIVE_DIR}/reference_images_full.zip'
os.makedirs(IMAGES_DIR, exist_ok=True)
print('Extracting reference_images_full.zip (this can take a few minutes)...')
with zipfile.ZipFile(full_zip, 'r') as z:
    z.extractall(IMAGES_DIR)

count = len([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
print(f'Extracted {count} images to {IMAGES_DIR}')

In [ ]:
# Step 5 — copy JSON files and fix Windows paths → Colab paths
import json, shutil
from pathlib import PureWindowsPath

shutil.copy(f'{DRIVE_DIR}/class_labels.json', '/content/class_labels.json')
shutil.copy(f'{DRIVE_DIR}/reference_images.json', '/content/reference_images.json')

with open('/content/reference_images.json') as f:
    ref = json.load(f)

fixed = {}
for key, paths in ref.items():
    path_list = paths if isinstance(paths, list) else [paths]
    fixed[key] = [f'{IMAGES_DIR}/{PureWindowsPath(p).name}' for p in path_list]

with open('/content/reference_images.json', 'w') as f:
    json.dump(fixed, f)

sample_keys = list(fixed.keys())[:2]
for k in sample_keys:
    print(k, '->', fixed[k][0])
print('Path rewrite done.')

# Optional: no_car_images.txt — skip if you haven't uploaded it (the build will
# simply not filter out any known-bad reference photos)
no_car_src = f'{DRIVE_DIR}/no_car_images.txt'
NO_CAR_PATH = '/content/no_car_images.txt'
if os.path.exists(no_car_src):
    shutil.copy(no_car_src, NO_CAR_PATH)
    print('no_car_images.txt copied.')
else:
    open(NO_CAR_PATH, 'w').close()
    print('no_car_images.txt not found in Drive — continuing without it.')

In [ ]:
# Step 6 — copy the clip_lora adapter if you uploaded one (keeps these embeddings
# consistent with what your local ai-service actually loads and serves)
LORA_SRC = f'{DRIVE_DIR}/clip_lora'
LORA_DIR = '/content/clip_lora'
if os.path.exists(LORA_SRC):
    shutil.copytree(LORA_SRC, LORA_DIR, dirs_exist_ok=True)
    print('clip_lora adapter copied:', os.listdir(LORA_DIR))
else:
    LORA_DIR = None
    print('No clip_lora folder found in Drive — using base CLIP.')

In [ ]:
# Step 7 — augmentation + helper logic
# (mirrors ai-service/src/model_loader.py exactly, so the cache this produces is a
# drop-in replacement for the one your local ai-service would build)
from pathlib import Path
from PIL import Image, ImageEnhance

_AUG_VERSION = 3   # v3: reference images are now YOLO-cropped before embedding,
                   # matching what predict() does at inference (see model_loader.py)
_MIN_IMAGES_FOR_HOLDOUT = 4
_EMBED_BATCH_SIZE = 64   # GPU can go much larger than the local CPU default of 16
_CAR_CLASSES = {2, 5, 7}   # COCO: car, bus, truck

def _augment(image):
    return [
        image,                                                      # original
        image.transpose(Image.FLIP_LEFT_RIGHT),                     # mirror
        ImageEnhance.Brightness(image).enhance(0.85),                # darker
        ImageEnhance.Brightness(image).enhance(1.15),                # brighter
        image.rotate(-8, expand=False, fillcolor=(128, 128, 128)),   # rotated left
        image.rotate(8, expand=False, fillcolor=(128, 128, 128)),    # rotated right
    ]

def _load_no_car_filenames(path):
    p = Path(path)
    if not p.exists():
        return set()
    return {line.strip() for line in p.read_text(encoding='utf-8').splitlines() if line.strip()}

In [ ]:
# Step 8 — load YOLO (for cropping) and CLIP (+ LoRA adapter if present) on GPU
from ultralytics import YOLO
from transformers import CLIPModel, CLIPProcessor
import torch.nn.functional as F
import numpy as np

_CLIP_MODEL = 'openai/clip-vit-large-patch14'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

yolo = YOLO('yolov8n.pt')   # auto-downloads; small model, runs fast even if it falls back to CPU

processor = CLIPProcessor.from_pretrained(_CLIP_MODEL)
base_clip = CLIPModel.from_pretrained(_CLIP_MODEL)

if LORA_DIR is not None:
    from peft import PeftModel
    clip_model = PeftModel.from_pretrained(base_clip, LORA_DIR)
    print('Loaded LoRA adapter.')
else:
    clip_model = base_clip

clip_model.eval()
clip_model.to(device)

def crop_car(image):
    # mirrors model_loader.py's _crop_car — largest detected vehicle bbox,
    # or the full image unchanged if nothing is detected
    results = yolo(image, verbose=False)
    boxes = [b for b in results[0].boxes if int(b.cls[0]) in _CAR_CLASSES]
    if not boxes:
        return image
    best = max(boxes, key=lambda b: (b.xyxy[0][2] - b.xyxy[0][0]) * (b.xyxy[0][3] - b.xyxy[0][1]))
    x1, y1, x2, y2 = (int(v) for v in best.xyxy[0])
    return image.crop((x1, y1, x2, y2))

def clip_embed_batch(images):
    enc = processor(images=images, return_tensors='pt')
    pixel_values = enc['pixel_values'].to(device)
    with torch.no_grad():
        out = clip_model.vision_model(pixel_values=pixel_values)
        feat = clip_model.visual_projection(out.pooler_output)
    vecs = F.normalize(feat, dim=-1).cpu().numpy()
    return vecs.astype(np.float32)

In [ ]:
# Step 9 — reserve held-out test images, filter no-car images, then batch-embed
# everything else (identical logic/order to model_loader.py's _build_index)
import json, time

with open('/content/class_labels.json', encoding='utf-8') as f:
    labels = json.load(f)
with open('/content/reference_images.json', encoding='utf-8') as f:
    ref_map = json.load(f)

no_car_names = _load_no_car_filenames('/content/no_car_images.txt')

per_key_paths = {}
for key, img_paths in ref_map.items():
    if key not in labels:
        continue
    valid = [Path(p) for p in img_paths if Path(p).exists() and Path(p).name not in no_car_names]
    if valid:
        per_key_paths[key] = valid

held_out_map = {}
for key, paths in per_key_paths.items():
    if len(paths) < _MIN_IMAGES_FOR_HOLDOUT:
        continue
    held_out_path = sorted(paths, key=lambda p: p.name)[-1]
    held_out_map[key] = str(held_out_path)
    per_key_paths[key] = [p for p in paths if p != held_out_path]

with open('/content/held_out_test_set.json', 'w', encoding='utf-8') as f:
    json.dump(held_out_map, f, indent='\t')
print(f'Reserved {len(held_out_map)} held-out test images.')

entries = [(key, p) for key, paths in per_key_paths.items() for p in paths]
n_augs = len(_augment(Image.new('RGB', (4, 4))))
print(f'Embedding {len(entries)} source images x {n_augs} augmentations …')

keys, source_ids, vecs = [], [], []
pending_imgs, pending_keys, pending_sources = [], [], []

def flush():
    if not pending_imgs:
        return
    batch_vecs = clip_embed_batch(pending_imgs)
    keys.extend(pending_keys)
    source_ids.extend(pending_sources)
    vecs.extend(batch_vecs)
    pending_imgs.clear(); pending_keys.clear(); pending_sources.clear()

t0 = time.time()
for i, (key, p) in enumerate(entries):
    try:
        img = Image.open(p).convert('RGB')
        cropped = crop_car(img)
        for aug in _augment(cropped):
            pending_imgs.append(aug)
            pending_keys.append(key)
            pending_sources.append(str(p))
            if len(pending_imgs) >= _EMBED_BATCH_SIZE:
                flush()
    except Exception as exc:
        print(f'Skipping {p}: {exc}')
    if (i + 1) % 500 == 0:
        elapsed = time.time() - t0
        print(f'  …{i+1}/{len(entries)} source images ({elapsed:.0f}s elapsed)')
flush()

emb_matrix = np.stack(vecs).astype(np.float32)
torch.save({
    'keys': keys,
    'embeddings': emb_matrix,
    'source_ids': source_ids,
    'source_count': len(entries),
    'aug_version': _AUG_VERSION,
}, '/content/reference_embeddings.pt')
print(f'Done in {time.time()-t0:.0f}s. Saved {len(keys)} embeddings.')

In [ ]:
# Step 10 — copy results back to Google Drive for download
import shutil, os
shutil.copy('/content/reference_embeddings.pt', f'{DRIVE_DIR}/reference_embeddings.pt')
shutil.copy('/content/held_out_test_set.json', f'{DRIVE_DIR}/held_out_test_set.json')
print('Saved to Drive:', DRIVE_DIR)
print('Files:', [f for f in os.listdir(DRIVE_DIR) if f in ('reference_embeddings.pt', 'held_out_test_set.json')])

## After running

1. Download **both** `reference_embeddings.pt` and `held_out_test_set.json` from
   `MyDrive/street_scout/` together (right-click each → Download).
2. Place both files at `ai-service/model/` on your PC, overwriting the old
   `reference_embeddings.pt` if present.
3. Restart the ai-service: `uvicorn src.main:app --reload --port 8000` — it will
   detect the cache matches (same source count and augmentation version) and load
   it directly instead of rebuilding.
4. Run `python train_classifier.py` once to retrain the classifier on the new
   embeddings (leak-free group split, ~2-5 minutes).
5. Run `python evaluate.py` to get an honest top-1/top-5 accuracy number against
   the held-out set.

**Important:** download both files from the *same* run and place them together —
`held_out_test_set.json` records which image was excluded from training for each
generation, and must match the embeddings that were actually built around it.